## **Agent Evaluation**

For RAG, we used the A->Q->A' setup:

* A = original answer in the FAQ
* Q = generated question from this answer
* A' = answer produced by our RAG system

For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.

We also save the trajectory. Here, the trajectory means only the tool calls the agent made before producing the final answer.

## Loading the data

Use the same ground truth questions:

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table:

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

## Running the agent

Reuse the ToyAIKit agent from module 01. It handles the agent loop and stores the full message history.

First, set up the model clients:

In [ ]:
# from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

# load_dotenv()
openai_client = OpenAI()

Define the search tool:

In [ ]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

Create the runner:

In [ ]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The result contains:

* *last_message*: the final response
* *all_messages*: the full message history
* *cost*: the cost of all LLM calls in this run

Run it for one ground truth question:

In [ ]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

Look at the full message history:

In [ ]:
result.all_messages

For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

Extract the function name and arguments:

In [ ]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

For this example:

In [ ]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

You should see something like this:

In [ ]:
# [
#     {
#         "name": "search",
#         "arguments": "{\"query\":\"own pace certificate at the end self-paced course certificate\"}"
#     }
# ]

Get the original answer:

In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

Save the A->Q->A' record and the trajectory:

In [ ]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

The *answer_agent* field is what we evaluate with the LLM judge. The *tool_calls* field lets the judge see how the agent got there.

## Processing multiple questions

Create a function that processes one ground truth record:

In [ ]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Run it for a small sample in parallel:

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

Turn it into a dataframe:

In [ ]:
df_agent = pd.DataFrame(agent_answers)

Calculate the total cost:

In [ ]:
df_agent["cost"].sum()

Save the results:

In [ ]:
df_agent.to_csv("data/agent-answers.csv", index=False)

Now we have the same A->Q->A' data as before, plus the tool calls for each agent run.

We generated this file for the course materials on May 29, 2026. The run used 50 ground truth questions. ToyAIKit tracks the agent cost for each run, so we can sum the cost column directly.

The total agent cost was $0.06993300, about 7 cents.

If you don't want to run the agent yourself, download the file we prepared:

In [ ]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget -O data/agent-answers.csv ${PREFIX}/04-evaluation/data/agent-answers.csv

Then load it:

In [ ]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

Our judge can look at both:

* whether answer_agent matches answer_orig
* whether the tool calls look reasonable for the question

This lets us evaluate the final answer and the agent behavior in one place.

## Judging answers and trajectories

A good trajectory is not just "many tool calls". A good trajectory uses the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

* The search query is relevant to the user question
* The query includes the important keywords from the question
* The agent avoids duplicate searches with the same arguments
* If it searches more than once, the next query is a useful refinement
* It usually uses 1 search call
* 2-3 calls can be okay for harder questions
* More than 3 search calls needs a clear reason
* The tool calls support the final answer
* The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

The judge instructions:

In [ ]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Define the judge function:

In [ ]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

Test it on one agent result:

In [ ]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

When the answer is bad, the trajectory score tells us whether the problem started with tool use. If the answer is bad but the trajectory is good, the model may have used the retrieved context poorly. If both are bad, the agent likely searched for the wrong thing. It may also have stopped too early.

## Running the agent judge

Run the judge for all agent answers:

In [ ]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

Use the same parallel helper:

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

Split the results:

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [ ]:
df_agent_eval = pd.DataFrame(agent_evaluations)

Calculate the judge cost from the token usage:

In [ ]:
calc_total_price(usages)

Check the answer scores:

In [ ]:
df_agent_eval["answer_score"].value_counts()

Check the trajectory scores:

In [ ]:
df_agent_eval["trajectory_score"].value_counts()

Save the judge results:

In [ ]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)

We generated this file for the course materials on May 29, 2026. The run judged 50 agent answers.

The answer scores were:

* Good: 45
* Bad: 5

The trajectory scores were:

* Good: 49
* Bad: 1

The judge token usage was:

* Input tokens: 29,228
* Output tokens: 6,984
* Cost with the prices above: $0.053349, about 5 cents

If you don't want to run the judge yourself, download the file we prepared:

In [ ]:
# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

# wget -O data/agent-evaluations.csv ${PREFIX}/04-evaluation/data/agent-evaluations.csv

## **Next Steps**

In this module, we covered evaluation at three levels:

1. Search evaluation: Hit Rate and MRR to measure retrieval quality
2. RAG evaluation: LLM-as-a-judge for answer quality
3. Agent evaluation: final answers plus tool-call trajectories

Evaluation is not a one-time activity. As you tune search parameters, switch models, or modify prompts, re-run evaluation. Make sure the system is getting better, not worse.

Evaluation is the most important part of building AI systems. It is also the most time-consuming. Only after evaluation can you be confident that your system works. Validate every change against your evaluation framework before going to production. This applies to prompt updates, model swaps, and agent modifications.

## From synthetic data to real data

The evaluation workflow in practice:

1. Start with synthetic data. Use an LLM to generate questions from your FAQ or documentation. This gives you a baseline without needing real users.
2. Tune the data generation. If the metrics look suspiciously good, the synthetic questions may be too close to the source text. Adjust the generation prompt to produce more realistic questions.
3. Deploy and collect real data. Once the system is in production, start collecting actual user queries and feedback.
4. Label real data. Have humans label whether the retrieved documents and generated answers are correct. This produces the most reliable ground truth.
5. Tune synthetic generation to match real data. Use the patterns from real queries to improve your synthetic data generator. The closer your synthetic data is to real data, the more useful the metrics become.

Nothing beats manual evaluation. Try the system yourself, think about edge cases, and collect examples of where it fails. This is especially important in the early stages when you don't have automated evaluation set up yet.

## Evaluation frameworks

For production systems, consider using evaluation frameworks that make it easier to manage test datasets, run evaluations, and track results:

* Ragas: a framework for evaluating RAG systems with metrics like faithfulness, answer relevance, and context precision
* DeepEval: provides built-in metrics for RAG evaluation including hallucination detection and answer relevance
* TruLens: instruments your LLM app and tracks quality metrics

These frameworks implement many of the concepts we covered here and add visualizations and experiment tracking.

## Monitoring

Online evaluation (monitoring) is what you do after deploying your system.

Key approaches:

* User feedback: thumbs up/down buttons to collect signal
* Logging: record queries, retrieved documents, and answers
* Dashboards: track metrics over time to spot degradation
* Alerts: get notified when metrics drop below a threshold

Monitoring is covered in more detail in module 05.